# Text summarization - Cosine similarity

# Preparing the environment

In [1]:
import re
import nltk
import string
import numpy as np
import networkx as nx
from nltk.cluster.util import cosine_distance

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/ankit/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/ankit/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
stopwords = nltk.corpus.stopwords.words('english')
print(stopwords)

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [4]:
def preprocess(text):
  formatted_text = text.lower()
  tokens = []
  for token in nltk.word_tokenize(formatted_text):
    tokens.append(token)
  tokens = [word for word in tokens if word not in stopwords and word not in string.punctuation]
  formatted_text = ' '.join(element for element in tokens)

  return formatted_text

In [5]:
original_text = """Artificial intelligence is human like intelligence.
                   It is the study of intelligent artificial agents.
                   Science and engineering to produce intelligent machines.
                   Solve problems and have intelligence.
                   Related to intelligent behavior.
                   Developing of reasoning machines.
                   Learn from mistakes and successes.
                   Artificial intelligence is related to reasoning in everyday situations."""
original_text = re.sub(r'\s+', ' ', original_text)
original_text

'Artificial intelligence is human like intelligence. It is the study of intelligent artificial agents. Science and engineering to produce intelligent machines. Solve problems and have intelligence. Related to intelligent behavior. Developing of reasoning machines. Learn from mistakes and successes. Artificial intelligence is related to reasoning in everyday situations.'

# Function to calculate similarity between sentences

- Link: https://en.wikipedia.org/wiki/Cosine_similarity
- Step by step calculations: https://janav.wordpress.com/2013/10/27/tf-idf-and-cosine-similarity/

In [6]:
original_sentences = [sentence for sentence in nltk.sent_tokenize(original_text)]
original_sentences

['Artificial intelligence is human like intelligence.',
 'It is the study of intelligent artificial agents.',
 'Science and engineering to produce intelligent machines.',
 'Solve problems and have intelligence.',
 'Related to intelligent behavior.',
 'Developing of reasoning machines.',
 'Learn from mistakes and successes.',
 'Artificial intelligence is related to reasoning in everyday situations.']

In [7]:
formatted_sentences = [preprocess(original_sentence) for original_sentence in original_sentences]
formatted_sentences

['artificial intelligence human like intelligence',
 'study intelligent artificial agents',
 'science engineering produce intelligent machines',
 'solve problems intelligence',
 'related intelligent behavior',
 'developing reasoning machines',
 'learn mistakes successes',
 'artificial intelligence related reasoning everyday situations']

In [8]:
def calculate_sentence_similarity(sentence1, sentence2):
  words1 = [word for word in nltk.word_tokenize(sentence1)]
  words2 = [word for word in nltk.word_tokenize(sentence2)]
  #print(words1)
  #print(words2)

  all_words = list(set(words1 + words2))
  #print(all_words)

  vector1 = [0] * len(all_words)
  vector2 = [0] * len(all_words)
  #print(vector1)
  #print(vector2)

  for word in words1: # Bag of words
    #print(word)
    vector1[all_words.index(word)] += 1
  for word in words2:
    vector2[all_words.index(word)] += 1

  #print(vector1)
  #print(vector2)

  return 1 - cosine_distance(vector1, vector2)

In [9]:
calculate_sentence_similarity(formatted_sentences[0], formatted_sentences[1])

np.float64(0.18898223650461365)

In [10]:
test = ['human', 'study', 'intelligence', 'agents', 'intelligent', 'artificial', 'like']
test.index('agents')

3

# Function to create the similarity matrix

In [11]:
# The higher the value, the greater the similarity between the sentences
# The more words in common, the greater the similarity

In [12]:
def calculate_similarity_matrix(sentences):
  similarity_matrix = np.zeros((len(sentences), len(sentences)))
  #print(similarity_matrix)
  for i in range(len(sentences)):
    for j in range(len(sentences)):
      if i == j:
        continue
      similarity_matrix[i][j] = calculate_sentence_similarity(sentences[i], sentences[j])
  return similarity_matrix

In [13]:
calculate_similarity_matrix(formatted_sentences)

array([[0.        , 0.18898224, 0.        , 0.43643578, 0.        ,
        0.        , 0.        , 0.46291005],
       [0.18898224, 0.        , 0.2236068 , 0.        , 0.28867513,
        0.        , 0.        , 0.20412415],
       [0.        , 0.2236068 , 0.        , 0.        , 0.25819889,
        0.25819889, 0.        , 0.        ],
       [0.43643578, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.23570226],
       [0.        , 0.28867513, 0.25819889, 0.        , 0.        ,
        0.        , 0.        , 0.23570226],
       [0.        , 0.        , 0.25819889, 0.        , 0.        ,
        0.        , 0.        , 0.23570226],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        ],
       [0.46291005, 0.20412415, 0.        , 0.23570226, 0.23570226,
        0.23570226, 0.        , 0.        ]])

# Function to summarize the texts

- Pagerank algorithm: https://en.wikipedia.org/wiki/PageRank


In [14]:
for i, score in enumerate(original_sentences):
  print(i, score)

0 Artificial intelligence is human like intelligence.
1 It is the study of intelligent artificial agents.
2 Science and engineering to produce intelligent machines.
3 Solve problems and have intelligence.
4 Related to intelligent behavior.
5 Developing of reasoning machines.
6 Learn from mistakes and successes.
7 Artificial intelligence is related to reasoning in everyday situations.


In [15]:
def summarize(text, number_of_sentences, percentage = 0):
  original_sentences = [sentence for sentence in nltk.sent_tokenize(text)]
  formatted_sentences = [preprocess(original_sentence) for original_sentence in original_sentences]
  similarity_matrix = calculate_similarity_matrix(formatted_sentences)
  #print(similarity_matrix)

  similarity_graph = nx.from_numpy_array(similarity_matrix)
  #print(similarity_graph.nodes)
  #print(similarity_graph.edges)

  scores = nx.pagerank(similarity_graph)
  #print(scores)
  ordered_scores = sorted(((scores[i], score) for i, score in enumerate(original_sentences)), reverse=True)
  #print(ordered_scores)

  if percentage > 0:
    number_of_sentences = int(len(formatted_sentences) * percentage)

  best_sentences = []
  for sentence in range(number_of_sentences):
    best_sentences.append(ordered_scores[sentence][1])

  return original_sentences, best_sentences, ordered_scores

In [16]:
original_sentences, best_sentences, scores = summarize(original_text, 3)

In [17]:
original_sentences

['Artificial intelligence is human like intelligence.',
 'It is the study of intelligent artificial agents.',
 'Science and engineering to produce intelligent machines.',
 'Solve problems and have intelligence.',
 'Related to intelligent behavior.',
 'Developing of reasoning machines.',
 'Learn from mistakes and successes.',
 'Artificial intelligence is related to reasoning in everyday situations.']

In [18]:
best_sentences

['Artificial intelligence is related to reasoning in everyday situations.',
 'Artificial intelligence is human like intelligence.',
 'It is the study of intelligent artificial agents.']

In [19]:
scores

[(0.21117425706958176,
  'Artificial intelligence is related to reasoning in everyday situations.'),
 (0.16732274004260206, 'Artificial intelligence is human like intelligence.'),
 (0.1455558879715914, 'It is the study of intelligent artificial agents.'),
 (0.12904008009949403, 'Related to intelligent behavior.'),
 (0.1275338306533556,
  'Science and engineering to produce intelligent machines.'),
 (0.10880185703391372, 'Solve problems and have intelligence.'),
 (0.08959232615044063, 'Developing of reasoning machines.'),
 (0.020979020979020983, 'Learn from mistakes and successes.')]

In [20]:
from IPython.core.display import HTML
def visualize(title, sentence_list, best_sentences):
  text = ''

  display(HTML(f'<h1>Summary - {title}</h1>'))
  for sentence in sentence_list:
    if sentence in best_sentences:
      text += ' ' + str(sentence).replace(sentence, f"<mark>{sentence}</mark>")
    else:
      text += ' ' + sentence
  display(HTML(f""" {text} """))

In [21]:
visualize('Artificial intelligence', original_sentences, best_sentences)

# Extracting texts from the Internet

In [31]:
# !pip install goose3

In [23]:
from goose3 import Goose
g = Goose()
url = 'https://en.wikipedia.org/wiki/Automatic_summarization'
article = g.extract(url)

In [24]:
article.cleaned_text

'There are two general approaches to automatic summarization: extraction and abstraction. Here, content is extracted from the original data, but the extracted content is not modified in any way. Examples of extracted content include key-phrases that can be used to "tag" or index a text document, or key sentences (including headings) that collectively comprise an abstract, and representative images or video segments, as stated above. For text, extraction is analogous to the process of skimming, where the summary (if available), headings and subheadings, figures, the first and last paragraphs of a section, and optionally the first and last sentences in a paragraph are read before one chooses to read the entire document in detail.[9] Other examples of extraction that include key sequences of text in terms of clinical relevance (including patient/problem, intervention, and outcome).[10] Abstractive summarization methods generate new text that did not exist in the original text.[11] This ha

In [25]:
original_sentences, best_sentences, scores = summarize(article.cleaned_text, 120, 0.2)

In [26]:
(120 / len(original_sentences)) * 100

260.8695652173913

In [27]:
original_sentences

['There are two general approaches to automatic summarization: extraction and abstraction.',
 'Here, content is extracted from the original data, but the extracted content is not modified in any way.',
 'Examples of extracted content include key-phrases that can be used to "tag" or index a text document, or key sentences (including headings) that collectively comprise an abstract, and representative images or video segments, as stated above.',
 'For text, extraction is analogous to the process of skimming, where the summary (if available), headings and subheadings, figures, the first and last paragraphs of a section, and optionally the first and last sentences in a paragraph are read before one chooses to read the entire document in detail.',
 '[9] Other examples of extraction that include key sequences of text in terms of clinical relevance (including patient/problem, intervention, and outcome).',
 '[10] Abstractive summarization methods generate new text that did not exist in the ori

In [28]:
best_sentences

['[48] Some video summarization systems assign importance scores to frames or segments to guide selection.',
 'Video summarization reduces a long video into a shorter version while keeping its important content, usually as keyframes or short segments, based on visual, motion, or audio cues.',
 'Representative segments preserve the main content of the original video, while diversity helps include different aspects of that content.',
 'Examples of extracted content include key-phrases that can be used to "tag" or index a text document, or key sentences (including headings) that collectively comprise an abstract, and representative images or video segments, as stated above.',
 "[50] Query-focused video summarization selects content according to a user's request, often expressed as a text query.",
 'In Machine Aided Human Summarization, extractive techniques highlight candidate passages for inclusion (to which the human adds or removes text).',
 '[10] Abstractive summarization methods gene

In [29]:
scores

[(0.039048268969003624,
  '[48] Some video summarization systems assign importance scores to frames or segments to guide selection.'),
 (0.038688242654020384,
  'Video summarization reduces a long video into a shorter version while keeping its important content, usually as keyframes or short segments, based on visual, motion, or audio cues.'),
 (0.03680911534746194,
  'Representative segments preserve the main content of the original video, while diversity helps include different aspects of that content.'),
 (0.03666992350003167,
  'Examples of extracted content include key-phrases that can be used to "tag" or index a text document, or key sentences (including headings) that collectively comprise an abstract, and representative images or video segments, as stated above.'),
 (0.036592082986868814,
  "[50] Query-focused video summarization selects content according to a user's request, often expressed as a text query."),
 (0.03109113628384387,
  'In Machine Aided Human Summarization, ext

In [30]:
visualize(article.title, original_sentences, best_sentences)